# Provider Tax Cap Analysis

**Medicaid Financing Workbench — Notebook 01**

This notebook walks through the provider-tax cap exposure analysis end to end: what question we're asking, how the arithmetic works, what the results look like, and what an analyst should be cautious about before citing any number.

---

## The question

The 2025 federal budget reconciliation law phases the provider-tax ceiling for expansion states from **6.0% to 3.5%** of net patient revenue, in 0.5-percentage-point annual steps from 2028 through 2032. It also freezes new or increased taxes immediately.

Provider taxes are a major financing mechanism for state Medicaid programs: a state taxes hospitals or other providers, uses the revenue as the nonfederal share, draws down a federal match, and (often) returns a portion to the providers. As of KFF's 2025–26 survey, 31 expansion states report a non-exempt provider tax above the new 3.5% ceiling.

The question this analysis answers: **which expansion states are most exposed, by how much, and on what timeline?**

## The formula

We don't directly observe the net-patient-revenue base on which the tax is levied; that would require state-level provider financial reports. So we proxy the at-risk revenue as the share of the nonfederal Medicaid share attributable to taxing above the target cap:

$$\text{exposed rate points} = \max(0,\ r - r_{\text{cap}})$$

$$\text{at-risk revenue} = \text{nonfederal share} \times \frac{\text{exposed rate points}}{r}$$

where $r$ is the state's current provider tax rate (as a percentage of net patient revenue) and $r_{\text{cap}}$ is the target ceiling (3.5%).

**What this number means:** it is the fraction of the state's current nonfederal Medicaid financing that is attributable to taxing above the 3.5% line — i.e., the revenue that must be replaced from another source or cut from the program once the phase-down completes.

The phase-down walks the ceiling down 0.5pp/year:

| Year | Ceiling |
|------|---------|
| 2028 | 5.5% |
| 2029 | 5.0% |
| 2030 | 4.5% |
| 2031 | 4.0% |
| 2032 | 3.5% |

Only expansion states are subject to the reduced cap; non-expansion states keep the 6.0% ceiling.

In [ ]:
import sys, pathlib
# Allow running from the notebooks/ directory
sys.path.insert(0, str(pathlib.Path('..').resolve() / 'src'))

import pandas as pd
import yaml

from mfw.etl.panel import build_panel, panel_summary
from mfw.analysis import provider_tax

print('Imports OK')

In [ ]:
# Load the panel and policy parameters
df = build_panel(prefer_live=False)   # prefer_live=True to attempt live CMS-64 pull
params = yaml.safe_load(open('../config/policy_parameters.yaml'))

summary = panel_summary(df)
print(f"Panel: {summary['n_states']} states | provenance={summary['data_provenance']}")
print(f"Expansion states: {summary['n_expansion']}")
df[['state','abbr','expansion','provider_tax_rate','nonfederal_share']].head()

## Run the analysis

In [ ]:
result = provider_tax.run(df, params)

print(f"States exposed (rate > 3.5%, expansion): {result['n_exposed']}")
print(f"National final-year gap:                 ${result['national_final_gap_billion']:.2f}B")
print(f"Top exposed state:                       {result['top_state']}")
print(f"Phase-down schedule: {result['schedule']}")

In [ ]:
# Display top exposed states as a DataFrame
exposed = [r for r in result['rows'] if r['exposed']]

top_df = pd.DataFrame(exposed)[[
    'state', 'provider_tax_rate', 'final_gap_millions',
    'gap_share_of_nonfederal_pct'
]].rename(columns={
    'provider_tax_rate': 'Tax Rate (%)',
    'final_gap_millions': 'Final-Year Gap ($M)',
    'gap_share_of_nonfederal_pct': 'Gap as % of Nonfed Share'
})

top_df = top_df.set_index('state')
top_df.style.format({
    'Tax Rate (%)': '{:.1f}%',
    'Final-Year Gap ($M)': '${:,.0f}M',
    'Gap as % of Nonfed Share': '{:.1f}%'
}).background_gradient(subset=['Final-Year Gap ($M)'], cmap='Reds')

## Year-by-year gap for the top 5 states

The gap opens gradually as the ceiling steps down. This shows the year-by-year at-risk revenue for the five most exposed states.

In [ ]:
top5 = exposed[:5]
years = sorted(result['schedule'].keys())

timeline_data = {
    r['state']: [r['gap_by_year'][yr] for yr in years]
    for r in top5
}
timeline_df = pd.DataFrame(timeline_data, index=years)
timeline_df.index.name = 'Year'
print('At-risk revenue by year ($M):')
timeline_df.style.format('${:,.0f}M')

## Render the chart inline

In [ ]:
import pathlib
from mfw.outputs.charts import chart_provider_tax_gap
from IPython.display import Image

out_dir = pathlib.Path('../outputs/charts')
chart_path = chart_provider_tax_gap(result, out_dir=out_dir)
Image(str(chart_path))

## So what — and the caveats

### The finding
The states with the largest final-year gaps are large expansion programs with provider tax rates near the current 6% ceiling. Collectively, the exposed states face a financing challenge that must be addressed — by replacing revenue, cutting provider rates, or reducing services — by 2032.

The timeline (2028–2032) gives states runway to plan, but also creates multi-year budget uncertainty. States with rates close to 3.5% have a smaller gap to close; states at 5.5–6% face a proportionally larger adjustment.

### Seed-data limitation
The nonfederal share figures used here are **illustrative seed values** anchored to KFF-published magnitudes. They are not CMS-64 actuals. The ranking of states and the directional finding (which states are most exposed) are likely to be stable, but the dollar figures should not be cited as official until replaced with live CMS-64 data via `mfw fetch --live`.

### Index/model parameters are analyst choices
The phase-down schedule, the nonfederal-share proxy, and the exposure formula are documented in `config/policy_parameters.yaml` and `src/mfw/analysis/provider_tax.py`. They reflect publicly available statutory language and a reasonable modeling choice. An analyst publishing from this workbench should:
1. Replace seed values with CMS-64 actuals
2. Document the exposure formula explicitly
3. Note that the figure represents at-risk provider-tax revenue above the cap, not the total projected spending reduction